# 04. GARCH(1,1) — Проверка устойчивости результатов

## Цель
Проверить, сохраняются ли выводы OLS-регрессий (ноутбук 03) при учёте **кластеризации волатильности** (volatility clustering) — характерной черты финансовых временных рядов.

## Спецификация GARCH(1,1)

**Уравнение среднего** (те же факторы, что в OLS):
$$r_{BTC,t} = \mu + \sum_k \beta_k X_{k,t} + \varepsilon_t$$

**Уравнение дисперсии:**
$$\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

где $\alpha + \beta < 1$ — условие ковариационной стационарности. Параметр $\alpha$ отражает реакцию волатильности на шоки, $\beta$ — её персистентность.

## Логика проверки устойчивости
- Если GARCH-оценки коэффициентов **совпадают по знаку и значимости** с OLS — выводы устойчивы
- Если картина меняется — необходимо обсудить причины

**Модели:**
- М1: крипто-факторы в уравнении среднего
- М2: внешние факторы в уравнении среднего  
- М3: все факторы в уравнении среднего

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch
import warnings
warnings.filterwarnings('ignore')

# Загружаем данные
df = pd.read_csv('../data/processed/merged_weekly.csv', index_col='date', parse_dates=True)

p1 = df.loc['2017-01-01':'2019-12-31']
p2 = df.loc['2020-01-01':'2025-03-01']

Y_VAR = 'r_btc'
CRYPTO_VARS   = ['r_btc_lag', 'log_volume_btc', 'google_trends']
EXTERNAL_VARS = ['r_sp500', 'vix', 'r_dxy']
ALL_VARS      = CRYPTO_VARS + EXTERNAL_VARS

var_labels = {
    'Const':          'Константа',
    'r_btc_lag':      'BTC доходность (лаг)',
    'log_volume_btc': 'Log объём BTC',
    'google_trends':  'Google Trends',
    'r_sp500':        'S&P 500 доходность',
    'vix':            'VIX',
    'r_dxy':          'DXY доходность',
    'omega':          'ω (GARCH const)',
    'alpha[1]':       'α (ARCH эффект)',
    'beta[1]':        'β (персистентность)',
}

print('Данные загружены.')
print(f'Полный период: {len(df)} недель | P1: {len(p1)} | P2: {len(p2)}')

## 1. Тест на ARCH-эффекты

Перед GARCH проверяем, есть ли в остатках OLS кластеризация волатильности (тест Энгла на ARCH-эффекты).

In [ ]:
import statsmodels.api as sm

# OLS остатки для теста
sample = df[ALL_VARS + [Y_VAR]].dropna()
y_ols = sample[Y_VAR]
X_ols = sm.add_constant(sample[ALL_VARS])
ols_resid = sm.OLS(y_ols, X_ols).fit().resid

# Тест Энгла на ARCH-эффекты (лаги 1, 4, 8)
print('Тест Энгла на ARCH-эффекты (H0: нет ARCH-эффектов):')
print(f'{"Лаг":<6} {"LM-стат.":<12} {"p-value":<12} {"Вывод"}')
for lag in [1, 4, 8]:
    lm, pval, _, _ = het_arch(ols_resid, nlags=lag)
    conclusion = '→ ARCH-эффекты есть' if pval < 0.05 else '→ нет'
    print(f'{lag:<6} {lm:<12.3f} {pval:<12.4f} {conclusion}')

## 2. Вспомогательные функции

In [ ]:
def run_garch(data, y_var, x_vars):
    """
    GARCH(1,1) с факторами в уравнении среднего.
    Доходности масштабируются на 100 для численной стабильности.
    """
    sample = data[[y_var] + x_vars].dropna()
    y = sample[y_var] * 100          # масштабирование
    X = sample[x_vars]

    am = arch_model(
        y, x=X,
        mean='ARX', lags=0,
        vol='GARCH', p=1, q=1,
        dist='normal'
    )
    res = am.fit(disp='off', options={'maxiter': 500})
    return res


def extract_garch_results(res, model_name=''):
    """
    Извлекает параметры из результатов GARCH.
    Возвращает две таблицы: уравнение среднего и уравнение дисперсии.
    """
    params  = res.params
    bse     = res.std_err
    tvals   = res.tvalues
    pvals   = res.pvalues

    mean_rows, var_rows = [], []
    var_params = {'omega', 'alpha[1]', 'beta[1]'}

    for var in params.index:
        pval  = float(pvals[var])
        stars = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.1 else ''))
        row = {
            'Переменная': var_labels.get(var, var),
            'Коэф.':    round(float(params[var]), 4),
            'Ст. ош.':  round(float(bse[var]),    4),
            't-стат.':  round(float(tvals[var]),   3),
            'p-value':  round(pval,                4),
            'Знач.':    stars,
        }
        if var in var_params:
            var_rows.append(row)
        else:
            mean_rows.append(row)

    df_mean = pd.DataFrame(mean_rows).set_index('Переменная')
    df_var  = pd.DataFrame(var_rows).set_index('Переменная')

    # Персистентность α + β
    alpha = float(params.get('alpha[1]', 0))
    beta  = float(params.get('beta[1]',  0))

    meta = pd.Series({
        'Log-Likelihood': round(res.loglikelihood, 2),
        'AIC':            round(res.aic,           2),
        'BIC':            round(res.bic,           2),
        'α + β':          round(alpha + beta,      4),
        'N':              int(res.nobs),
    }, name=model_name)

    return df_mean, df_var, meta


print('Функции определены.')

## 3. GARCH(1,1) — Полный период (2017–2025)

In [ ]:
print('Обучение моделей (может занять ~10–20 сек)...')

g1 = run_garch(df, Y_VAR, CRYPTO_VARS)
g2 = run_garch(df, Y_VAR, EXTERNAL_VARS)
g3 = run_garch(df, Y_VAR, ALL_VARS)

print('Готово.')

In [ ]:
for res, label in [(g1, 'М1: Крипто-факторы'), (g2, 'М2: Внешние факторы'), (g3, 'М3: Все факторы')]:
    mean_tbl, var_tbl, meta = extract_garch_results(res, label)
    print(f'\n=== GARCH(1,1) — {label} — Полный период ===')
    print('Уравнение среднего:')
    display(mean_tbl)
    print('Уравнение дисперсии:')
    display(var_tbl)
    print(f'AIC={meta["AIC"]:.2f}  BIC={meta["BIC"]:.2f}  α+β={meta["α + β"]:.4f}  N={meta["N"]}')

## 4. GARCH(1,1) — Анализ по подпериодам

In [ ]:
for period_name, data in [('2017–2019', p1), ('2020–2025', p2)]:
    print(f'\n{'='*60}')
    print(f'ПОДПЕРИОД: {period_name}')
    print('='*60)
    for x_vars, label in [
        (CRYPTO_VARS,   'М1: Крипто-факторы'),
        (EXTERNAL_VARS, 'М2: Внешние факторы'),
    ]:
        r = run_garch(data, Y_VAR, x_vars)
        mean_tbl, var_tbl, meta = extract_garch_results(r, label)
        print(f'\n--- {label} ---')
        display(mean_tbl)
        print(f'α+β={meta["α + β"]:.4f}  AIC={meta["AIC"]:.2f}  N={meta["N"]}')

## 5. Сравнительная таблица: OLS vs GARCH

Для каждого фактора сравниваем коэффициент и значимость из OLS (ноутбук 03) и GARCH(1,1).

In [ ]:
def comparison_table(x_vars, model_label):
    """Сравнение OLS vs GARCH по трём периодам."""
    periods = [('2017–2025', df), ('2017–2019', p1), ('2020–2025', p2)]
    rows = {}

    for period_name, data in periods:
        sample = data[[Y_VAR] + x_vars].dropna()

        # OLS
        y_o = sample[Y_VAR]
        X_o = sm.add_constant(sample[x_vars])
        ols = sm.OLS(y_o, X_o).fit()
        nw_lags = int(np.floor(4 * (len(sample)/100)**(2/9)))
        ols_r = ols.get_robustcov_results(cov_type='HAC', maxlags=nw_lags)

        # GARCH
        g = run_garch(data, Y_VAR, x_vars)

        for var in x_vars:
            key = var_labels.get(var, var)
            if key not in rows:
                rows[key] = {}

            # OLS коэф.
            names_ols = list(ols_r.model.exog_names)
            if var in names_ols:
                i = names_ols.index(var)
                p_ols = float(np.asarray(ols_r.pvalues)[i])
                s_ols = '***' if p_ols<0.01 else ('**' if p_ols<0.05 else ('*' if p_ols<0.1 else ''))
                c_ols = float(np.asarray(ols_r.params)[i])
                rows[key][f'OLS {period_name}'] = f'{c_ols:.4f}{s_ols}'

            # GARCH коэф. (масштаб /100)
            if var in g.params.index:
                p_g = float(g.pvalues[var])
                s_g = '***' if p_g<0.01 else ('**' if p_g<0.05 else ('*' if p_g<0.1 else ''))
                c_g = float(g.params[var]) / 100
                rows[key][f'GARCH {period_name}'] = f'{c_g:.4f}{s_g}'

    table = pd.DataFrame(rows).T
    # Упорядочиваем колонки
    col_order = [f'{m} {p}' for p in ['2017–2025','2017–2019','2020–2025'] for m in ['OLS','GARCH']]
    table = table[[c for c in col_order if c in table.columns]]
    print(f'\n=== {model_label}: OLS vs GARCH(1,1) ===')
    display(table)

comparison_table(CRYPTO_VARS,   'Модель 1: Крипто-факторы')
comparison_table(EXTERNAL_VARS, 'Модель 2: Внешние факторы')

## 6. Условная волатильность BTC

In [ ]:
# Условная волатильность из М3 (все факторы, полный период)
cond_vol = g3.conditional_volatility / 100  # обратное масштабирование

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Доходность BTC
ax1.plot(df.index[-len(cond_vol):], df[Y_VAR].iloc[-len(cond_vol):],
         color='steelblue', linewidth=0.8, alpha=0.9)
ax1.axvline(pd.Timestamp('2020-01-01'), color='red', linestyle='--', alpha=0.7, label='01.2020')
ax1.set_title('Недельная доходность BTC')
ax1.set_ylabel('Доходность')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Условная волатильность
vol_index = df.index[-len(cond_vol):]
ax2.plot(vol_index, cond_vol.values, color='darkorange', linewidth=1.0)
ax2.axvline(pd.Timestamp('2020-01-01'), color='red', linestyle='--', alpha=0.7, label='01.2020')
ax2.fill_between(vol_index, cond_vol.values, alpha=0.2, color='darkorange')
ax2.set_title('Условная волатильность GARCH(1,1) — М3 (все факторы)')
ax2.set_ylabel('σ_t (усл. откл.)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../data/processed/garch_cond_vol.png', dpi=100, bbox_inches='tight')
plt.show()
print('График сохранён: data/processed/garch_cond_vol.png')

## 7. Сводная таблица GARCH-параметров дисперсии

In [ ]:
garch_variance_summary = []

configs = [
    ('М1: Крипто', CRYPTO_VARS,   df,  'Полный'),
    ('М2: Внешние', EXTERNAL_VARS, df,  'Полный'),
    ('М3: Все',     ALL_VARS,      df,  'Полный'),
    ('М1: Крипто', CRYPTO_VARS,   p1,  '2017–2019'),
    ('М2: Внешние', EXTERNAL_VARS, p1,  '2017–2019'),
    ('М1: Крипто', CRYPTO_VARS,   p2,  '2020–2025'),
    ('М2: Внешние', EXTERNAL_VARS, p2,  '2020–2025'),
]

for model_label, x_vars, data, period in configs:
    r = run_garch(data, Y_VAR, x_vars)
    p = r.params
    pv = r.pvalues
    alpha = float(p.get('alpha[1]', np.nan))
    beta  = float(p.get('beta[1]',  np.nan))
    pa    = float(pv.get('alpha[1]', np.nan))
    pb    = float(pv.get('beta[1]',  np.nan))
    sa    = '***' if pa<0.01 else ('**' if pa<0.05 else ('*' if pa<0.1 else ''))
    sb    = '***' if pb<0.01 else ('**' if pb<0.05 else ('*' if pb<0.1 else ''))
    garch_variance_summary.append({
        'Модель': model_label,
        'Период': period,
        'α (ARCH)': f'{alpha:.4f}{sa}',
        'β (GARCH)': f'{beta:.4f}{sb}',
        'α + β': round(alpha + beta, 4),
        'AIC': round(r.aic, 2),
    })

var_summary_df = pd.DataFrame(garch_variance_summary)
print('=== Параметры уравнения дисперсии GARCH(1,1) ===')
display(var_summary_df)
print('\nα + β близко к 1 → высокая персистентность волатильности (типично для крипто)')

## 8. Выводы: устойчивость результатов

In [ ]:
print('=== ВЫВОДЫ ПО УСТОЙЧИВОСТИ РЕЗУЛЬТАТОВ ===')
print()

# Сравниваем значимость S&P 500 в OLS и GARCH по подпериодам
for period_name, data in [('2017–2019', p1), ('2020–2025', p2)]:
    g = run_garch(data, Y_VAR, EXTERNAL_VARS)
    sp500_p = float(g.pvalues.get('r_sp500', 1.0))
    sp500_c = float(g.params.get('r_sp500', 0)) / 100
    stars   = '***' if sp500_p<0.01 else ('**' if sp500_p<0.05 else ('*' if sp500_p<0.1 else 'незначим'))
    print(f'S&P 500 в GARCH М2 ({period_name}): коэф.={sp500_c:.4f}, p={sp500_p:.4f} {stars}')

print()
for period_name, data in [('2017–2019', p1), ('2020–2025', p2)]:
    g = run_garch(data, Y_VAR, CRYPTO_VARS)
    lag_p = float(g.pvalues.get('r_btc_lag', 1.0))
    lag_c = float(g.params.get('r_btc_lag', 0)) / 100
    stars  = '***' if lag_p<0.01 else ('**' if lag_p<0.05 else ('*' if lag_p<0.1 else 'незначим'))
    print(f'BTC momentum в GARCH М1 ({period_name}): коэф.={lag_c:.4f}, p={lag_p:.4f} {stars}')

print()
print('Интерпретация:')
print('  • Если S&P 500 значим в P2 в GARCH — результаты OLS устойчивы (H2 подтверждается)')
print('  • Если ARCH-параметры значимы — GARCH модель обоснована')
print('  • α + β < 1 — ковариационная стационарность соблюдена')